# Compartment taxonomy

A `Property` is a group of mutually exclusive traits. A `PropertyMap` is the
table of compartments those properties create. Queries are selectors
(`age["0-4"]`, `state["I"] & ~sev["severe"]`) that compile to integer index arrays.

Ragged maps are allowed: a property may be absent on some compartments. Absent
is Kleene *unknown*, so both `severity["mild"]` and `~severity["mild"]` skip
unstratified rows. Use `.absent()` / `.present()` to reach them.

The bars are the claim to check. Severity exists only on `I`, so that
state has twice as many compartments as `S` or `R` (12 rows, not a full
3 × 3 × 2 = 18). The second figure is the Kleene split: mild, not-mild,
and absent together cover every row and do not overlap.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
sev = Property("severity", ("mild", "severe"))

pm = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(sev, where=state["I"])
)

assert pm.size == 12  # S×3 + I×3×2 + R×3
assert pm.n_properties == 3
pm

counts = pd.Series(
    {
        "S": pm.select(state["S"]).size,
        "I, mild": pm.select(state["I"] & sev["mild"]).size,
        "I, severe": pm.select(state["I"] & sev["severe"]).size,
        "R": pm.select(state["R"]).size,
    }
)
assert int(counts.sum()) == pm.size
counts.to_frame("compartments").plot.bar(
    title="Severity exists only on I",
    labels={"index": "stratum", "value": "compartments"},
)



## Selectors skip what they do not know

`state["I"] & age["0-4"]` is the two infected compartments in that band
(mild and severe). `~severity["severe"]` does **not** mean "everyone else":
rows that were never given a severity are unknown, so they drop out of both
`severe` and `~severe`. `.absent()` is how you name those rows — here, all
of `S` and `R`.


In [ ]:
infected_young = pm.select(state["I"] & age["0-4"])
assert infected_young.size == 2  # mild and severe

not_severe = pm.select(age[("0-4", "5-9")] & ~sev["severe"])
assert set(not_severe.tolist()).isdisjoint(pm.select(sev["severe"]).tolist())

unstratified = pm.select(sev.absent())
assert set(unstratified.tolist()) == set(pm.select(state["S"] | state["R"]).tolist())


## A partition, and the three-way severity split

`partition(age)` names every age band and together those index arrays cover
the map. The figure is the same fact for severity: mild, not-mild, and
absent are disjoint and exhaust the 12 compartments. Not-mild is only the
severe infectious rows; susceptibles and recovered sit in *absent*.


In [ ]:
by_age = pm.partition(age)
assert set(t.name for t in by_age) == {"0-4", "5-9", "10+"}
assert sum(idx.size for idx in by_age.values()) == pm.size

mild = set(pm.select(sev["mild"]).tolist())
not_mild = set(pm.select(~sev["mild"]).tolist())
absent = set(pm.select(sev.absent()).tolist())
assert mild | not_mild | absent == set(range(pm.size))
assert mild.isdisjoint(not_mild) and mild.isdisjoint(absent)

split = pd.Series(
    {
        "mild": len(mild),
        "not mild": len(not_mild),
        "severity absent": len(absent),
    }
)
assert int(split.sum()) == pm.size
split.to_frame("compartments").plot.bar(
    title="Mild, not-mild, and absent cover the map without overlap",
    labels={"index": "severity query", "value": "compartments"},
)
